# 60 — M5 one-step validation

M5 専用ノートブックです。凍結済み M4 parent を検証し、M5 proof primitives / CPU tests /
orchestrator を実行します。

固定 run ID:

- parent: `M4-20260720T021737Z-b9c9514fed11`
- current: `M5-20260720T051020Z-c3800fceaa80`

未解決の deterministic residual を 0 とみなして `ONE_STEP_CERTIFIED` を出しません。

In [ ]:
import importlib.util, subprocess, sys
for package in ('pytest', 'sympy'):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
import pytest
print('pytest:', pytest.__version__)
print('python:', sys.version.split()[0])
if sys.version_info < (3, 11):
    raise RuntimeError('Python 3.11+ is required for M5.')

In [ ]:
from pathlib import Path
import os, sys

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
    Path('/storage') / BUNDLE_NAME,
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'validated_4d_su2_rg_codex_design.md').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('M5 project root was not found.')
PERSIST_ROOT = Path(os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M4_RUN_ID', 'M4-20260720T021737Z-b9c9514fed11')
os.environ.setdefault('VALIDATED_RG_M5_RUN_ID', 'M5-20260720T051020Z-c3800fceaa80')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)
print('PERSIST_ROOT =', PERSIST_ROOT)

In [ ]:
from src.m5_parent import verify_accepted_m4_parent
from src.m5_status import M4_PARENT_RUN_ID_FROZEN, M5_RUN_ID_FROZEN

parent_run_id = os.environ['VALIDATED_RG_M4_RUN_ID']
m5_run_id = os.environ['VALIDATED_RG_M5_RUN_ID']
if parent_run_id != M4_PARENT_RUN_ID_FROZEN:
    raise RuntimeError(f'Unexpected M4 parent run id: {parent_run_id}')
if m5_run_id != M5_RUN_ID_FROZEN:
    raise RuntimeError(f'Unexpected M5 run id: {m5_run_id}')
PARENT_EVIDENCE = verify_accepted_m4_parent(PROJECT_ROOT, PERSIST_ROOT, parent_run_id)
{
    'parent_run_id': parent_run_id,
    'm5_run_id': m5_run_id,
    'parent_hashes': PARENT_EVIDENCE.hashes,
    'open_for_M5': PARENT_EVIDENCE.bound_ledger['open_for_M5'],
    'tensor_count': len(PARENT_EVIDENCE.tensors),
}

In [ ]:
# Phase B: proof-primitive CPU suite must pass before numerical certification work.
# Notebook cwd is often /notebooks; always resolve tests from PROJECT_ROOT.
import os
import pytest

os.chdir(PROJECT_ROOT)
test_files = [
    PROJECT_ROOT / 'tests/test_one_step_certificate.py',
    PROJECT_ROOT / 'tests/test_independent_one_step_verifier.py',
    PROJECT_ROOT / 'tests/test_m5_orchestrator.py',
    PROJECT_ROOT / 'tests/test_m5_parent.py',
    PROJECT_ROOT / 'tests/test_m5_obligations.py',
]
missing = [str(path) for path in test_files if not path.is_file()]
if missing:
    raise RuntimeError(
        'M5 test files are missing from PROJECT_ROOT. '
        'Pull the latest main (commit with src/ and tests/), then rerun. '
        f'Missing: {missing}'
    )
rc = pytest.main([
    '-q',
    f'--rootdir={PROJECT_ROOT}',
    *[str(path) for path in test_files],
])
if rc != 0:
    raise RuntimeError(f'M5 CPU proof-primitive suite failed with code {rc}')
print('M5 CPU proof-primitive suite: PASS')
print('cwd =', Path.cwd())
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
from src.m5_config import default_m5_config
from src.m5_orchestrator import create_or_resume_m5

m5_config = default_m5_config(
    parent_m4_run_id=parent_run_id,
    run_id=m5_run_id,
    mode='paperspace',
)
m5_orchestrator = create_or_resume_m5(PERSIST_ROOT, m5_config, PROJECT_ROOT)
M5_RESULT = m5_orchestrator.run_until_checkpoint()
M5_RESULT

## M5 実行境界

`paperspace` mode は次を実行します。

1. 凍結 M4 parent の immutable 検証
2. parent inventory / schema mapping
3. 8 個の M4→M5 handoff obligations の評価（`reports/M5_obligation_report.json`）
4. すべて RIGOROUS なら live `one_step_certificate/` を組み立て、独立 verifier を通し、
   `reports/M5_acceptance.json` を書いて M6 を開ける

以前 OPEN にしていた 4 項は、未証明の 0 埋めを避けるための意図的停止でした。
現在は限定スコープ（singleton 入力球・凍結 cutoff/rank・M1 telescoping lift・
打ち切り sector 網羅）で閉じます。連続極限は主張しません。

`M5_COMPLETE` 後は `VALIDATED_RG_M5_RUN_ID` を設定して `70_m6_multistep_certificate.ipynb` へ。
